# One-shot on Kaggle: all 58 Who&When runs

This notebook runs **one-shot** (`methods/one_shot.py`) on the test split (all 58 runs) on Kaggle's GPU, with the
same code, the same Ollama version and the same model files as the Mac. Its rules are in `informatics.md`, section 3.

**Before you run it**
1. **Push your code to GitHub first** (branch `SC.V1`). The notebook clones the project from GitHub, so it runs
   whatever is pushed, not what is on your Mac.
2. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
3. In the notebook's right-hand panel (**Settings**): **Accelerator: GPU T4 x2** and **Internet: On**.
4. **Only if the GitHub repo is private:** **Add-ons → Secrets → Add a new secret** named `GITHUB_TOKEN`, with a
   GitHub token that can read the repo. Switch it on for this notebook.
5. **Save Version → Save & Run All (Commit)**. The notebook then runs in the background, even with the browser
   closed, for up to 12 hours.

**What you get** (right-hand panel, **Output**, once it has finished)
- `one_shot_logs_<date>.zip`: one log per run (`run_<number>_<time>.md`) and `summary_<time>.csv`, one line per run.
- The same files unzipped under `can-debate-be-used-to-reduce-human-review-time/runs/one_shot/<date>/`.

**On the Mac** the same notebook also runs: it skips the Kaggle-only steps and uses this project folder and its
running Ollama server.

## 1. Get the project and its code

In [ ]:
import csv
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path

REPOSITORY_ADDRESS = "github.com/shoimya/can-debate-be-used-to-reduce-human-review-time.git"
BRANCH = "SC.V1"
KAGGLE_PROJECT_FOLDER = Path("/kaggle/working/can-debate-be-used-to-reduce-human-review-time")

# Here we check whether this notebook runs on Kaggle: every Kaggle notebook has the folder /kaggle/working.
RUNNING_ON_KAGGLE = Path("/kaggle/working").exists()


def github_address_with_token_if_any() -> str:
    """Return the repository address, with the Kaggle Secret GITHUB_TOKEN in it if one was added (for a private repo)."""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        return f"https://{token}@{REPOSITORY_ADDRESS}"
    except Exception:
        return f"https://{REPOSITORY_ADDRESS}"


def find_or_clone_project_folder() -> Path:
    """On Kaggle, clone (or update) the project from GitHub; on a Mac, find the project folder above this notebook."""
    if not RUNNING_ON_KAGGLE:
        # Here we go up from the notebook's folder (Notebooks/) until we reach the folder that holds settings.py.
        for folder in [Path.cwd(), *Path.cwd().parents]:
            if (folder / "settings.py").exists():
                return folder
        raise FileNotFoundError("No project folder (a folder with settings.py) was found above this notebook.")

    if KAGGLE_PROJECT_FOLDER.exists():
        subprocess.run(["git", "-C", str(KAGGLE_PROJECT_FOLDER), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, github_address_with_token_if_any(),
                        str(KAGGLE_PROJECT_FOLDER)], check=True)
    return KAGGLE_PROJECT_FOLDER


# Here we move into the project folder and let Python import its files from there, from methods/ and from setup/.
PROJECT_FOLDER = find_or_clone_project_folder()
os.chdir(PROJECT_FOLDER)
for code_folder in [PROJECT_FOLDER, PROJECT_FOLDER / "methods", PROJECT_FOLDER / "setup"]:
    sys.path.insert(0, str(code_folder))

# Here we print which commit of the code this run uses, so every result can be traced back to its code.
last_commit = subprocess.run(["git", "log", "-1", "--format=%h %s (%ci)"], capture_output=True, text=True)
print(f"Project folder: {PROJECT_FOLDER}")
print(f"Code version:   {last_commit.stdout.strip()}")

## 2. Settings for this run

Change only the lines in this cell. Everything else (the model, the prompt, the number of calls, the temperature
and the seed) comes from the project's own files, so the Mac and Kaggle run the same method.

In [ ]:
# Here we choose which runs to do: "pilot" is run 36 only (a quick check), "test" is all 58 runs.
SPLIT_TO_RUN = "test"

# Here we can redo only some runs, by their numbers, for example [30]; an empty list means every run in the split.
ONLY_RUN_NUMBERS = []

# Here we name the Ollama version and the model files the Mac uses, so Kaggle runs exactly the same ones.
# A model's ID is the first 12 characters of its digest, as `ollama list` shows it.
OLLAMA_VERSION_ON_THE_MAC = "0.35.1"
MODEL_IDS_ON_THE_MAC = {"qwen3.5:9b": "56671c2ab938", "qwen3.5:2b": "0689d44085e0"}

import data
import one_shot
import runV2
import settings
from start_ollama_server import ollama_server_is_running, start_ollama_server
from stop_ollama_server import stop_ollama_server

# Here we show the settings this run uses: the project's settings, then one-shot's own.
settings.print_settings()
print()
print("One-shot settings (methods/one_shot.py):")
print(f"    Calls allowed per run: {one_shot.DefaultNoOfCalls}")
print(f"    Temperature:           {one_shot.Temperature}")
print(f"    Base seed:             {one_shot.Seed} (call 1 uses {one_shot.Seed + 1}, call 2 uses {one_shot.Seed + 2}, ...)")
print(f"    Split:                 {SPLIT_TO_RUN}" + (f", only runs {ONLY_RUN_NUMBERS}" if ONLY_RUN_NUMBERS else ""))

## 3. Check the GPU (Kaggle only)

Without a GPU the model would run on the CPU and take days, so the notebook stops here with a clear message.

In [ ]:
def check_the_gpu_is_switched_on() -> None:
    """On Kaggle, stop with a clear message if no GPU is switched on; on the Mac, there is nothing to check."""
    if not RUNNING_ON_KAGGLE:
        print("On the Mac: Ollama uses the Mac's own GPU, nothing to check here.")
        return

    # Here we look for nvidia-smi, the program that comes with the GPU driver; without it there is no GPU.
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("No GPU found. In the notebook's Settings, set Accelerator to 'GPU T4 x2', then run again.")

    # Here we print each GPU's name and memory, so the output shows what the run used.
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], check=True)


check_the_gpu_is_switched_on()

## 4. Install Ollama (Kaggle only)

Kaggle starts every session from a clean machine, so Ollama is installed each time, in exactly the Mac's version.
Two small tools are installed first if missing: `zstd` (the Ollama installer unpacks with it) and `lsof` (the
project's stop script finds the server with it).

In [ ]:
def install_missing_system_tools() -> None:
    """Install zstd and lsof with apt-get, but only the ones that are missing."""
    missing_tools = [tool for tool in ["zstd", "lsof"] if shutil.which(tool) is None]
    if not missing_tools:
        print("zstd and lsof are already installed.")
        return

    print(f"Installing {', '.join(missing_tools)} ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *missing_tools], check=True)


def installed_ollama_version() -> str | None:
    """Return the installed Ollama version, for example "0.35.1", or None if Ollama is not installed."""
    if shutil.which("ollama") is None:
        return None

    # Here we ask Ollama for its version. Its last word is always this program's own version, for example
    # "Warning: client version is 0.35.1" when no server is running.
    version_output = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    all_output = version_output.stdout + version_output.stderr
    return all_output.strip().split()[-1]


def install_ollama(version: str) -> None:
    """Install the given Ollama version with Ollama's own install script, unless that version is already installed."""
    if installed_ollama_version() == version:
        print(f"Ollama {version} is already installed.")
        return

    # Here we pass the version to the install script, so Kaggle gets exactly the Mac's version.
    install_environment = dict(os.environ)
    install_environment["OLLAMA_VERSION"] = version

    # Here "pipefail" makes the step fail if the download of the install script fails, instead of silently doing nothing.
    install_command = "set -o pipefail; curl -fsSL https://ollama.com/install.sh | sh"
    subprocess.run(["bash", "-c", install_command], env=install_environment, check=True)


if RUNNING_ON_KAGGLE:
    install_missing_system_tools()
    install_ollama(OLLAMA_VERSION_ON_THE_MAC)

# Here we stop on Kaggle if the version is not the Mac's, because a different version can answer differently.
ollama_version_in_use = installed_ollama_version()
print(f"Ollama version in use: {ollama_version_in_use} (Mac: {OLLAMA_VERSION_ON_THE_MAC})")
if RUNNING_ON_KAGGLE and ollama_version_in_use != OLLAMA_VERSION_ON_THE_MAC:
    raise RuntimeError(f"Ollama {ollama_version_in_use} is installed, but the Mac uses {OLLAMA_VERSION_ON_THE_MAC}.")

## 5. Start the server and download the models

The server runs on this project's port (11500). On Kaggle the models go to `/tmp/ollama_models` (not saved with the
notebook), and download in a few minutes. The notebook then checks they are the same model files as on the Mac.

In [ ]:
def print_end_of_server_log(number_of_lines: int = 30) -> None:
    """Print the last lines of the Ollama server's own log, where it says why it did not start."""
    server_log_file = settings.MODELS_FOLDER / "ollama-server.log"
    if server_log_file.exists():
        print("\n".join(server_log_file.read_text().splitlines()[-number_of_lines:]))


def model_ids_on_this_server() -> dict:
    """Return each model's ID on this project's server, for example {"qwen3.5:9b": "56671c2ab938"}."""
    with urllib.request.urlopen(settings.OLLAMA_ADDRESS + "/api/tags", timeout=10) as response:
        models = json.load(response)["models"]
    # Here we keep the first 12 characters of each digest, the ID `ollama list` shows.
    return {model["name"]: model["digest"][:12] for model in models}


def warn_if_models_differ_from_the_mac() -> None:
    """Say for each model whether it is the same file as on the Mac, and warn loudly if it is not."""
    ids_here = model_ids_on_this_server()
    for model_name, id_on_the_mac in MODEL_IDS_ON_THE_MAC.items():
        id_here = ids_here.get(model_name)
        if id_here == id_on_the_mac:
            print(f"{model_name}: same model file as the Mac ({id_here}).")
        else:
            print(f"WARNING: {model_name} is not the Mac's model file (Mac: {id_on_the_mac}, here: {id_here}). "
                  "Answers may differ from the Mac's.")


# Here we start this project's server, unless it is already running, and remember whether we started it,
# so the last cell stops only a server this notebook started.
started_here = start_ollama_server()
if not ollama_server_is_running():
    print_end_of_server_log()
    raise RuntimeError("This project's Ollama server did not start; its log is printed above.")

# Here we download any model that is missing from settings.MODELS_FOLDER (download_models() skips the others).
settings.download_models()
warn_if_models_differ_from_the_mac()

## 6. Read the runs

In [ ]:
# Here we download any missing Who&When runs (about 4 MB) and check that every run's Decisive step exists.
all_runs = data.download_who_and_when()
usable_runs = data.remove_bad_runs(all_runs)

# Here we read the split file the whole team shares, so run numbers match the Mac's logs.
split_entries = json.loads((data.SPLITS_FOLDER / f"{SPLIT_TO_RUN}.json").read_text())

# Here we keep only the chosen run numbers, if any were chosen, and stop if one of them is not in the split.
if ONLY_RUN_NUMBERS:
    numbers_in_split = [entry["number"] for entry in split_entries]
    numbers_not_in_split = [number for number in ONLY_RUN_NUMBERS if number not in numbers_in_split]
    if numbers_not_in_split:
        raise ValueError(f"Runs {numbers_not_in_split} are not in the {SPLIT_TO_RUN} split.")
    split_entries = [entry for entry in split_entries if entry["number"] in ONLY_RUN_NUMBERS]

# Here we read each chosen run from its JSON file, keeping its split entry (number, ID, file) next to it.
runs_to_do = []
for entry in split_entries:
    run = data.read_who_and_when_file(data.WHO_AND_WHEN_FOLDER / entry["file"])
    runs_to_do.append((entry, run))

total_steps = sum(len(run.steps) for entry, run in runs_to_do)
print(f"Runs to do: {len(runs_to_do)} ({total_steps} steps in total).")

## 7. Run one-shot on every run

Each run's log is written the moment that run finishes, and the summary file is rewritten after every run, so a
session that stops early keeps every finished run. A run that fails is reported in full and the notebook goes on
with the next one.

In [ ]:
# Here we put the logs in one folder per day, as methods/one_shot.py does: runs/one_shot/<date>/.
LOG_FOLDER = PROJECT_FOLDER / "runs" / "one_shot" / time.strftime("%Y-%m-%d")

# Here we name the summary after the time this whole run started, so a second run on the same day keeps both.
SUMMARY_FILE = LOG_FOLDER / f"summary_{time.strftime('%H-%M-%S')}.csv"


def gpu_share_of_the_debater_model() -> str:
    """Return how much of the debater model sits on the GPU, for example "100% GPU", from the server's loaded models."""
    with urllib.request.urlopen(settings.OLLAMA_ADDRESS + "/api/ps", timeout=10) as response:
        loaded_models = json.load(response)["models"]

    for model in loaded_models:
        if model["name"] == settings.DEBATER_MODEL:
            # Here we compare the model's memory on the GPU with its whole size; anything below 100% is on the CPU.
            share_on_gpu = round(100 * model["size_vram"] / model["size"])
            return f"{share_on_gpu}% GPU"
    return "not loaded"


def run_one_shot_and_describe_the_result(entry: dict, run: data.Trajectory) -> dict:
    """Run one-shot on one run and return its summary line: answer, label, hits, minutes, GPU share and any error."""
    start_time = time.time()
    try:
        answer = one_shot.run_one_shot(run, entry["number"], LOG_FOLDER)
        error_text = ""
    except Exception as error:
        # Here we keep going after a failed run, so one failure does not throw away hours of finished runs.
        # The full error is printed here and kept in the summary; a failed run has no log, only finished runs do.
        traceback.print_exc()
        answer = None
        error_text = repr(error)

    # Here we score the answer the same way the log does: exactly right, and within the tolerance (±2 steps).
    has_answer = answer is not None
    exactly_right = has_answer and answer == run.decisive_step
    within_tolerance = has_answer and abs(answer - run.decisive_step) <= runV2.HIT_TOLERANCE_IN_STEPS

    return {
        "run_number": entry["number"],
        "file": entry["file"],
        "steps": len(run.steps),
        "answer": answer if has_answer else "",
        "decisive_step": run.decisive_step,
        "exactly_right": exactly_right,
        "within_tolerance": within_tolerance,
        "minutes": round((time.time() - start_time) / 60, 1),
        "gpu_share": gpu_share_of_the_debater_model(),
        "error": error_text,
    }


def write_summary_file(summary_rows: list[dict]) -> None:
    """Write every finished run's summary line to the CSV file, replacing the file written after the last run."""
    LOG_FOLDER.mkdir(parents=True, exist_ok=True)
    with open(SUMMARY_FILE, "w", newline="") as summary_file:
        summary_writer = csv.DictWriter(summary_file, fieldnames=list(summary_rows[0].keys()))
        summary_writer.writeheader()
        summary_writer.writerows(summary_rows)


# Here we remember when this run started, to find this run's logs and to show the time taken.
RUN_START_TIME = time.time()
summary_rows = []

for position, (entry, run) in enumerate(runs_to_do, start=1):
    print(f"[{position}/{len(runs_to_do)}] Run {entry['number']} ({entry['file']}, {len(run.steps)} steps) ...", flush=True)

    summary_row = run_one_shot_and_describe_the_result(entry, run)
    summary_rows.append(summary_row)
    write_summary_file(summary_rows)

    # Here we print this run's result and how long the whole run has taken so far.
    minutes_so_far = (time.time() - RUN_START_TIME) / 60
    print(f"    answer {summary_row['answer']}, label {summary_row['decisive_step']}, "
          f"exact: {summary_row['exactly_right']}, within ±{runV2.HIT_TOLERANCE_IN_STEPS}: {summary_row['within_tolerance']}, "
          f"{summary_row['minutes']} min, {summary_row['gpu_share']} | {minutes_so_far:.0f} min so far", flush=True)

    # Here we warn loudly if part of the model ran on the CPU, which makes every call many times slower.
    if not summary_row["error"] and summary_row["gpu_share"] != "100% GPU":
        print(f"    WARNING: the model is {summary_row['gpu_share']}, so part of it runs on the CPU.", flush=True)

## 8. Summary

In [ ]:
def logs_written_by_this_run() -> list[Path]:
    """Return the run logs this run wrote: the ones in the log folder changed since this run started."""
    return [log_file for log_file in sorted(LOG_FOLDER.glob("run_*.md")) if log_file.stat().st_mtime >= RUN_START_TIME]


def print_summary(summary_rows: list[dict]) -> None:
    """Print how many runs finished, the hit rates, the runs with no answer, the failed runs and any log warnings."""
    finished_rows = [row for row in summary_rows if not row["error"]]
    failed_rows = [row for row in summary_rows if row["error"]]
    number_finished = len(finished_rows)

    # Here we count the hits among the finished runs; a run with no answer counts as a miss.
    number_exactly_right = sum(row["exactly_right"] for row in finished_rows)
    number_within_tolerance = sum(row["within_tolerance"] for row in finished_rows)
    number_without_answer = sum(row["answer"] == "" for row in finished_rows)

    print(f"Runs finished: {number_finished} of {len(summary_rows)}")
    if number_finished:
        print(f"Exactly right:    {number_exactly_right} of {number_finished} ({100 * number_exactly_right / number_finished:.1f}%)")
        print(f"Within ±{runV2.HIT_TOLERANCE_IN_STEPS} steps: {number_within_tolerance} of {number_finished} "
              f"({100 * number_within_tolerance / number_finished:.1f}%)")
        print(f"No answer (no call gave a valid step): {number_without_answer}")
    # Here we repeat the random-guess baseline from data_analysis.ipynb (all 58 runs), to read the numbers against.
    print("For comparison, random guessing on all 58 runs: 4.2% exactly right, 17.8% within ±2 steps.")
    print(f"Time taken: {(time.time() - RUN_START_TIME) / 60:.0f} minutes")

    for row in failed_rows:
        print(f"FAILED: run {row['run_number']} ({row['file']}): {row['error']}")

    # Here we list every log with a warning, because a cut prompt or a cut reply makes that run's answer less trustworthy.
    for log_file in logs_written_by_this_run():
        log_text = log_file.read_text()
        for warning in ["PROMPT MAY HAVE BEEN CUT", "REPLY WAS CUT"]:
            if warning in log_text:
                print(f"WARNING in {log_file.name}: {warning} ({log_text.count(warning)} calls)")

    print(f"Summary file: {SUMMARY_FILE}")


print_summary(summary_rows)

## 9. Stop the server and pack the logs

In [ ]:
# Here we stop the server only if this notebook started it, never one someone else is using.
if started_here:
    stop_ollama_server()

# Here we zip this day's log folder into /kaggle/working, so all logs and the summary download as one file.
if RUNNING_ON_KAGGLE:
    archive_path = shutil.make_archive(f"/kaggle/working/one_shot_logs_{LOG_FOLDER.name}", "zip", LOG_FOLDER)
    print(f"Logs packed into: {archive_path}")
else:
    print(f"Logs are in: {LOG_FOLDER}")